# Dữ đoán giá nhà

## Chuẩn bị dữ liệu

### Khai báo thư viện

In [14]:
import os, sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
    
from IPython import display
import warnings
warnings.filterwarnings('ignore')

### Khai báo các tham số

In [15]:
params_cfg = {
    'action': globals().get('action', 'main_feat01'),
    'seed': globals().get('seed', 42),
    'exps_dir': globals().get('exps_dir', '../../exps'),
    'exp_name': globals().get('exp_name', 'house_prices_baseline'),
    'data_dir': globals().get('data_dir', '../../data/house-prices'),
    'verbose': globals().get('verbose', False),
    'k_folds': globals().get('k_folds', 5),
}

params_cfg.update(**{'save_dir': os.path.abspath(f'{params_cfg["exps_dir"]}/{params_cfg["exp_name"]}')})

for v in params_cfg:
    print(f'+ {v}: {params_cfg[v]}')

+ action: main_feat01
+ seed: 42
+ exps_dir: ../../exps
+ exp_name: house_prices_baseline
+ data_dir: ../../data/house-prices
+ verbose: False
+ k_folds: 5
+ save_dir: d:\Data\Code\Python\ML WS\exps\house_prices_baseline


### Tải dữ liệu train, test

In [16]:
df_train = pd.read_csv(f'{params_cfg['data_dir']}/train.csv')
df_test = pd.read_csv(f'{params_cfg['data_dir']}/test.csv')

display.display(df_train.head())
display.display(df_test.head())

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal


## Preprocessing

In [17]:
def preprocess_data(df, is_train=True, is_debug=False, **kwargs):
    """
    Tiền xử lý dữ liệu cho bài toán dự đoán giá nhà
    
    Parameters:
    - df: DataFrame đầu vào
    - is_train: True nếu là training data (có cột SalePrice), False nếu là test data
    - is_debug: True để in thông tin debug
    """
    
    # Tạo bản copy để không ảnh hưởng đến dataframe gốc
    df_output = df.copy()
    
    # Tách features và target (nếu là training data)
    if is_train:
        X = df_output.drop('SalePrice', axis=1)
        y = df_output['SalePrice']
    else:
        X = df_output
    
    # Phân loại columns
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

    if is_debug:
        print(f"Numerical features: {len(numeric_features)}")
        print(f"Categorical features: {len(categorical_features)}")
        print(f"Total features: {len(numeric_features) + len(categorical_features)}")

    # Xử lý missing values cho numerical features
    for col in numeric_features:
        if df_output[col].isnull().any():
            if is_debug:
                print(f"Filling missing values in numerical column: {col}")
            df_output[col] = df_output[col].fillna(df_output[col].median())

    # Xử lý missing values cho categorical features
    for col in categorical_features:
        if df_output[col].isnull().any():
            if is_debug:
                print(f"Filling missing values in categorical column: {col}")
            df_output[col] = df_output[col].fillna(df_output[col].mode()[0])

    # Thêm cột target trở lại nếu là training data
    if is_train:
        df_output['SalePrice'] = y

    if is_debug:
        print(f"Missing values after preprocessing: {df_output.isnull().sum().sum()}")
        print(f"Final shape: {df_output.shape}")

    return df_output

In [18]:
print("\nPreprocessing training data...")
print("----------------------------")
df_train_processed = preprocess_data(df_train, is_train=True, is_debug=True)
print("\nPreprocessing test data...")
print("----------------------------")
df_test_processed = preprocess_data(df_test, is_train=False, is_debug=True)

df_all = pd.concat([df_train_processed, df_test_processed], axis=0).reset_index(drop=True)
df_encoded = pd.get_dummies(df_all)
df_train_encoded = df_encoded.iloc[:len(df_train_processed), :].reset_index(drop=True)
df_test_encoded = df_encoded.iloc[len(df_train_processed):, :].reset_index(drop=True).drop('SalePrice', axis=1)


Preprocessing training data...
----------------------------
Numerical features: 37
Categorical features: 43
Total features: 80
Filling missing values in numerical column: LotFrontage
Filling missing values in numerical column: MasVnrArea
Filling missing values in numerical column: GarageYrBlt
Filling missing values in categorical column: Alley
Filling missing values in categorical column: MasVnrType
Filling missing values in categorical column: BsmtQual
Filling missing values in categorical column: BsmtCond
Filling missing values in categorical column: BsmtExposure
Filling missing values in categorical column: BsmtFinType1
Filling missing values in categorical column: BsmtFinType2
Filling missing values in categorical column: Electrical
Filling missing values in categorical column: FireplaceQu
Filling missing values in categorical column: GarageType
Filling missing values in categorical column: GarageFinish
Filling missing values in categorical column: GarageQual
Filling missing value

## Model training

### Baseline

In [19]:
numeric_features = df_train_processed.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df_train_processed.select_dtypes(include=['object', 'category']).columns.tolist()

base_models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'Random Forest': RandomForestRegressor(random_state=params_cfg['seed']),
    'Gradient Boosting': GradientBoostingRegressor(random_state=params_cfg['seed']),
    'XGBoost': XGBRegressor(random_state=params_cfg['seed']),
    'LightGBM': LGBMRegressor(random_state=params_cfg['seed'], verbose=-1),
    'SVR': SVR(),
    'K-Neighbors': KNeighborsRegressor(),
    'CatBoost': CatBoostRegressor(random_state=params_cfg['seed'], verbose=False, allow_writing_files=False, train_dir=None)
}

In [20]:
kf = KFold(n_splits=params_cfg['k_folds'], shuffle=True, random_state=params_cfg['seed'])

for model_name, model in base_models.items():
    print(f"Training {model_name}...")
    for fold, (train_index, val_index) in enumerate(kf.split(df_train_encoded.drop('Id', axis=1))):
        
        X_train = df_train_encoded.iloc[train_index].drop('SalePrice', axis=1)
        y_train = df_train_encoded.iloc[train_index]['SalePrice']  # Series
        X_val = df_train_encoded.iloc[val_index].drop('SalePrice', axis=1)
        y_val = df_train_encoded.iloc[val_index]['SalePrice']     # Series
        
        pipeline = Pipeline(steps=[
            ('scaler', StandardScaler()),  # Chỉ scale numerical
            ('model', model)
        ])
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)
        
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        r2 = r2_score(y_val, y_pred)
        
        print(f"Fold {fold+1} - RMSE: {rmse:.4f}, R2: {r2:.4f}")

Training Linear Regression...
Fold 1 - RMSE: 75805926802537200.0000, R2: -749190892690730431545344.0000
Fold 2 - RMSE: 466349078361518080.0000, R2: -31986012524193943237165056.0000
Fold 3 - RMSE: 122440540785386512.0000, R2: -2713593289310831877554176.0000
Fold 4 - RMSE: 243613354303400864.0000, R2: -9451581885215313605165056.0000
Fold 5 - RMSE: 850407259072001280.0000, R2: -138360690152470192142155776.0000
Training Ridge...
Fold 1 - RMSE: 28442.2066, R2: 0.8945
Fold 2 - RMSE: 28708.7468, R2: 0.8788
Fold 3 - RMSE: 54661.7535, R2: 0.4592
Fold 4 - RMSE: 34599.8888, R2: 0.8093
Fold 5 - RMSE: 27384.6513, R2: 0.8565
Training Lasso...
Fold 1 - RMSE: 28307.4222, R2: 0.8955
Fold 2 - RMSE: 28727.2355, R2: 0.8786
Fold 3 - RMSE: 54607.6627, R2: 0.4602
Fold 4 - RMSE: 34558.2017, R2: 0.8098
Fold 5 - RMSE: 27405.3133, R2: 0.8563
Training ElasticNet...
Fold 1 - RMSE: 30732.4534, R2: 0.8769
Fold 2 - RMSE: 30612.0314, R2: 0.8622
Fold 3 - RMSE: 45365.3263, R2: 0.6275
Fold 4 - RMSE: 29502.8670, R2: 0.861

### Tuning

In [21]:
param_grids = {
    'Linear Regression': {
        'model__fit_intercept': [True, False],
        'model__copy_X': [True, False],
        'model__positive': [True, False]  # Force positive coefficients
    },
    
    'Ridge': {
        'model__alpha': uniform(0.1, 10.0),
        'model__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg']
    },
    
    'Lasso': {
        'model__alpha': uniform(0.001, 1.0),
        'model__max_iter': randint(1000, 5000)
    },
    
    'ElasticNet': {
        'model__alpha': uniform(0.001, 1.0),
        'model__l1_ratio': uniform(0.1, 0.9),
        'model__max_iter': randint(1000, 5000)
    },
    
    'Random Forest': {
        'model__n_estimators': randint(100, 500),
        'model__max_depth': [None, 10, 15, 20],
        'model__min_samples_split': randint(2, 10),
        'model__min_samples_leaf': randint(1, 5),
        'model__max_features': ['auto', 'sqrt', 'log2']
    },
    
    'Gradient Boosting': {
        'model__n_estimators': randint(100, 500),
        'model__learning_rate': uniform(0.01, 0.3),
        'model__max_depth': randint(3, 8),
        'model__min_samples_split': randint(2, 10),
        'model__min_samples_leaf': randint(1, 5),
        'model__subsample': uniform(0.6, 0.4)  # 0.6 to 1.0
    },
    
    'XGBoost': {
        'model__n_estimators': randint(100, 500),
        'model__learning_rate': uniform(0.01, 0.3),
        'model__max_depth': randint(3, 10),
        'model__min_child_weight': randint(1, 10),
        'model__subsample': uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4),
        'model__reg_alpha': uniform(0, 1),
        'model__reg_lambda': uniform(0, 1)
    },
    
    'LightGBM': {
        'model__n_estimators': randint(100, 500),
        'model__learning_rate': uniform(0.01, 0.3),
        'model__max_depth': randint(3, 10),
        'model__num_leaves': randint(20, 100),
        'model__subsample': uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4),
        'model__reg_alpha': uniform(0, 1),
        'model__reg_lambda': uniform(0, 1)
    },
    
    'SVR': {
        'model__C': uniform(0.1, 10),
        'model__gamma': ['scale', 'auto'] + list(uniform(0.001, 0.1).rvs(5)),
        'model__kernel': ['rbf', 'linear', 'poly']
    },
    
    'K-Neighbors': {
        'model__n_neighbors': randint(3, 15),
        'model__weights': ['uniform', 'distance'],
        'model__p': [1, 2]  # 1: manhattan, 2: euclidean
    },
    
    'CatBoost': {
        'model__iterations': randint(100, 500),
        'model__learning_rate': uniform(0.01, 0.3),
        'model__depth': randint(4, 10),
        'model__l2_leaf_reg': uniform(1, 10),
        'model__subsample': uniform(0.6, 0.4)
    }
}

In [22]:
rmse_results = {}
best_models = {}

for model_name in base_models.keys():
    print(f"\nTuning {model_name}...")
    
    # Lấy model và param grid
    model = base_models[model_name]
    param_grid = param_grids[model_name]
    
    # Tạo pipeline
    pipeline = Pipeline(steps=[
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    
    # RandomizedSearchCV
    random_search = RandomizedSearchCV(
        pipeline,
        param_grid,
        n_iter=10,
        cv=3,
        scoring='neg_mean_squared_error',
        random_state=42,
        n_jobs=-1
    )
    
    # Fit toàn bộ data
    X = df_train_encoded.drop(columns=['Id', 'SalePrice'], axis=1)
    y = df_train_encoded['SalePrice']
    
    random_search.fit(X, y)
    
    # In kết quả
    best_rmse = np.sqrt(-random_search.best_score_)
    print(f"- Best RMSE: {best_rmse:.4f}")
    print(f"- Best params: {random_search.best_params_}")

    # Lưu kết quả
    rmse_results[model_name] = best_rmse
    best_models[model_name] = random_search.best_estimator_


Tuning Linear Regression...
- Best RMSE: 531880580543064896.0000
- Best params: {'model__positive': False, 'model__fit_intercept': True, 'model__copy_X': True}

Tuning Ridge...
- Best RMSE: 33760.5814
- Best params: {'model__alpha': 7.319987722668247, 'model__solver': 'svd'}

Tuning Lasso...
- Best RMSE: 33998.5664
- Best params: {'model__alpha': 0.8334426408004217, 'model__max_iter': 3853}

Tuning ElasticNet...
- Best RMSE: 31662.8110
- Best params: {'model__alpha': 0.6128528947223795, 'model__l1_ratio': 0.22554447458683766, 'model__max_iter': 4547}

Tuning Random Forest...
- Best RMSE: 31838.5349
- Best params: {'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 3, 'model__n_estimators': 291}

Tuning Gradient Boosting...
- Best RMSE: 27295.9319
- Best params: {'model__learning_rate': 0.05286004537658223, 'model__max_depth': 5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 6, 'model__n_estimators': 357, 'model_

In [23]:
os.makedirs(params_cfg['save_dir'], exist_ok=True)

# Lưu models
for model_name, model in best_models.items():
    filename = f"{model_name.replace(' ', '_').lower()}_best_model.pkl"
    filepath = os.path.join(params_cfg['save_dir'], filename)
    joblib.dump(model, filepath)
    print(f"Saved: {filename}")

Saved: linear_regression_best_model.pkl
Saved: ridge_best_model.pkl
Saved: lasso_best_model.pkl
Saved: elasticnet_best_model.pkl
Saved: random_forest_best_model.pkl
Saved: gradient_boosting_best_model.pkl
Saved: xgboost_best_model.pkl
Saved: lightgbm_best_model.pkl
Saved: svr_best_model.pkl
Saved: k-neighbors_best_model.pkl
Saved: catboost_best_model.pkl


In [24]:
# Tìm model tốt nhất
best_model_name = min(rmse_results, key=rmse_results.get)
best_model = best_models[best_model_name]

print(f"\nBEST MODEL: {best_model_name}")
print(f"Best RMSE: {rmse_results[best_model_name]:.4f}")

# Hiển thị ranking tất cả models
print("\nMODEL RANKING:")
for model_name, rmse in sorted(rmse_results.items(), key=lambda x: x[1]):
    print(f"{model_name:20}: {rmse:.4f}")


BEST MODEL: CatBoost
Best RMSE: 25966.7727

MODEL RANKING:
CatBoost            : 25966.7727
LightGBM            : 27105.1170
Gradient Boosting   : 27295.9319
XGBoost             : 27867.1880
ElasticNet          : 31662.8110
Random Forest       : 31838.5349
Ridge               : 33760.5814
Lasso               : 33998.5664
K-Neighbors         : 39925.2242
SVR                 : 48203.3873
Linear Regression   : 531880580543064896.0000


## Submit

In [25]:
best_model.fit(
    df_train_encoded.drop(columns=['Id', 'SalePrice'], axis=1),
    df_train_encoded['SalePrice']
)

y_test_pred = best_model.predict(df_test_encoded.drop('Id', axis=1))
df_submission = pd.DataFrame({
    'Id': df_test_encoded['Id'],
    'SalePrice': y_test_pred
})

df_submission.to_csv(f"{params_cfg['save_dir']}/submission_best_model.csv", index=False)